# Live pre-Bedrock parity check

This notebook runs the **legacy Ebola collection path** and the **new `mm_pipeline` path**, then compares their prepared master datasets before Bedrock extraction. It does not change production code.

It makes real OPoint and public-web requests and may update caches used by the legacy fetchers. The two runs happen sequentially, so live-source drift and the legacy OPoint helper's call-time window can produce legitimate differences. Treat the report as a differential audit, not as a guarantee of byte-for-byte equality.

In [1]:
from datetime import UTC, datetime, timedelta
from pathlib import Path
import json
import logging
import os
import sys

import pandas as pd
from dotenv import load_dotenv

DAYS_BACK = 7
USE_OPOINT = True
USE_HEALTH = True
STORED_SEARCH_ID = 101838198

def find_pipeline_root(start: Path) -> Path:
    for base in (start, *start.parents):
        if (base / 'src' / 'mm_pipeline').is_dir():
            return base
        if (base / 'mm_pipeline' / 'src' / 'mm_pipeline').is_dir():
            return base / 'mm_pipeline'
    raise FileNotFoundError('Could not locate the mm_pipeline project root')

def find_legacy_root(pipeline_root: Path) -> Path:
    candidates = [
        pipeline_root.parent,  # Current nested layout.
        pipeline_root.parent / 'MM_Ebola_Tracker',  # Sibling layout after copying.
    ]
    for candidate in candidates:
        if (candidate / 'src' / 'opoint_functions.py').exists():
            return candidate
    raise FileNotFoundError(
        'Could not locate the legacy project. Set LEGACY_ROOT explicitly in this cell.'
    )

MM_PIPELINE_ROOT = find_pipeline_root(Path.cwd().resolve())
LEGACY_ROOT = find_legacy_root(MM_PIPELINE_ROOT)

print('New pipeline:', MM_PIPELINE_ROOT)
print('Legacy project:', LEGACY_ROOT)

New pipeline: C:\Users\TimSilman\Documents\other_projects\MM_Ebola_Tracker\mm_pipeline
Legacy project: C:\Users\TimSilman\Documents\other_projects\MM_Ebola_Tracker


In [3]:
for env_path in (LEGACY_ROOT / '.env', MM_PIPELINE_ROOT / '.env'):
    if env_path.exists():
        load_dotenv(env_path, override=False)
        print('Loaded environment from:', env_path)

# The legacy project used TOKEN; the new configuration uses OPOINT_TOKEN.
if os.getenv('TOKEN') and not os.getenv('OPOINT_TOKEN'):
    os.environ['OPOINT_TOKEN'] = os.environ['TOKEN']
if os.getenv('OPOINT_TOKEN') and not os.getenv('TOKEN'):
    os.environ['TOKEN'] = os.environ['OPOINT_TOKEN']

if USE_OPOINT and not os.getenv('OPOINT_TOKEN'):
    raise RuntimeError('Set TOKEN or OPOINT_TOKEN in .env before running the live check.')

# Make both codebases importable without installing or modifying either one.
for path in (MM_PIPELINE_ROOT / 'src', LEGACY_ROOT):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from src.fetchers import enrich_item_text, fetch_all_sources
from src.opoint_functions import collect_opoint, standardise_opoint
from src.utils import load_sources
from mm_pipeline.config import load_project_config
from mm_pipeline.models import CollectionWindow
from mm_pipeline.pipeline import run_pipeline

logging.basicConfig(level=logging.INFO, format='%(levelname)s %(name)s: %(message)s')
logger = logging.getLogger('pre_bedrock_parity')

END_TIME = datetime.now(UTC).replace(microsecond=0)
START_TIME = END_TIME - timedelta(days=DAYS_BACK)
WINDOW = CollectionWindow(start=START_TIME, end=END_TIME)
RUN_DIR = MM_PIPELINE_ROOT / 'parity_output' / END_TIME.strftime('%Y%m%dT%H%M%SZ')
RUN_DIR.mkdir(parents=True, exist_ok=False)

print(f'Comparison window: {START_TIME.isoformat()} to {END_TIME.isoformat()}')
print('Report directory:', RUN_DIR)

Loaded environment from: C:\Users\TimSilman\Documents\other_projects\MM_Ebola_Tracker\.env
Comparison window: 2026-10-20T16:34:21+00:00 to 2026-10-27T16:34:21+00:00
Report directory: C:\Users\TimSilman\Documents\other_projects\MM_Ebola_Tracker\mm_pipeline\parity_output\20261027T163421Z


## 1. Run the legacy pre-Bedrock path

This combines the calls previously split between `src.test_ingestion` and `01_master_data_pull.ipynb`: fetch and enrich public-health items, call OPoint, standardise, filter, deduplicate, and remove records with no analysis text.

In [5]:
COMMON_COLUMNS = [
    'platform', 'source_id', 'published_at', 'country', 'language',
    'source_name', 'author', 'headline', 'content', 'summary', 'url', 'search_id',
]
legacy_frames = []
legacy_failures = []
legacy_source_health = []

if USE_OPOINT:
    raw_opoint = collect_opoint(
        token=os.environ['TOKEN'],
        stored_search_id=STORED_SEARCH_ID,
        days_back=DAYS_BACK,
    )
    legacy_frames.append(standardise_opoint(raw_opoint))

if USE_HEALTH:
    legacy_items, legacy_failures, legacy_source_health = fetch_all_sources(
        sources=load_sources(),
        logger=logger.getChild('legacy.health'),
        start_date=START_TIME.date(),
        end_date=END_TIME.date(),
    )
    enrichment_logger = logger.getChild('legacy.enrichment')
    enriched_count = 0
    for item in legacy_items:
        if 'Google News' not in item.source:
            enrich_item_text(item, enrichment_logger)
            enriched_count += 1
            if enriched_count % 50 == 0:
                logger.info('Legacy enrichment progress: %s articles', enriched_count)
    logger.info('Legacy enrichment complete: %s articles', enriched_count)
    legacy_frames.append(pd.DataFrame([
        {
            'platform': 'health_source',
            'source_id': item.url or item.title,
            'published_at': item.published_at,
            'country': '',
            'language': 'en',
            'source_name': item.source,
            'author': '',
            'headline': item.title,
            'content': item.extracted_text or '',
            'summary': item.summary or '',
            'url': item.url,
            'search_id': '',
        }
        for item in legacy_items
    ]))

if not legacy_frames:
    raise ValueError('Enable at least one legacy source group')

for frame in legacy_frames:
    for column in COMMON_COLUMNS:
        if column not in frame.columns:
            frame[column] = ''
legacy_raw = pd.concat([frame[COMMON_COLUMNS] for frame in legacy_frames], ignore_index=True)
legacy_raw['article_id'] = (
    legacy_raw['platform'].fillna('unknown').astype(str).str.strip()
    + '_' + legacy_raw['source_id'].fillna('').astype(str).str.strip()
)
content_text = legacy_raw['content'].fillna('').astype(str).str.strip()
summary_text = legacy_raw['summary'].fillna('').astype(str).str.strip()
legacy_raw['analysis_text'] = content_text.where(content_text.ne(''), summary_text)

health_keywords = [
    'ebola', 'ebola virus', 'ebola disease', 'drc',
    'democratic republic of the congo', 'congo', 'kivu', 'north kivu',
    'south kivu', 'ituri', 'kasai', 'uganda', 'marburg',
    'viral haemorrhagic fever', 'viral hemorrhagic fever',
]
health_mask = legacy_raw['platform'].eq('health_source')
health_text = (
    legacy_raw['headline'].fillna('') + ' '
    + legacy_raw['content'].fillna('') + ' '
    + legacy_raw['summary'].fillna('')
).str.lower()
keep_health = health_text.str.contains('|'.join(health_keywords), case=False, na=False, regex=True)
legacy_filtered = legacy_raw[(~health_mask) | keep_health].copy()
legacy_duplicate_count = int(legacy_filtered['article_id'].duplicated().sum())
legacy_df = legacy_filtered.drop_duplicates('article_id', keep='last').copy()
legacy_df = legacy_df[legacy_df['analysis_text'].fillna('').astype(str).str.strip().ne('')].copy()
legacy_path = RUN_DIR / 'legacy_master_data.jsonl'
legacy_df.to_json(legacy_path, orient='records', lines=True, force_ascii=False, date_format='iso')

print(f'Legacy prepared records: {len(legacy_df)}')
print(f'Legacy source failures: {len(legacy_failures)}')
display(legacy_df['platform'].value_counts(dropna=False).rename('records').to_frame())

INFO pre_bedrock_parity.legacy.health: Fetched 0 RSS items from CDC Newsroom
INFO pre_bedrock_parity.legacy.health: Fetched 300 RSS items from CDC MMWR
INFO pre_bedrock_parity.legacy.health: Fetched 22 RSS items from CDC Travel Health Notices
INFO pre_bedrock_parity.legacy.health: Fetched 0 HTML list items from WHO Disease Outbreak News
INFO pre_bedrock_parity.legacy.health: Fetched 10 RSS items from WHO Regional Office for Africa
INFO pre_bedrock_parity.legacy.health: Fetched 0 RSS items from Africa CDC
INFO pre_bedrock_parity.legacy.health: Fetched 20 NCDC news items from Nigeria Centre for Disease Control
INFO pre_bedrock_parity.legacy.health: Fetched 10 RSS items from ECDC News
INFO pre_bedrock_parity: Legacy enrichment progress: 50 articles
INFO pre_bedrock_parity: Legacy enrichment progress: 100 articles
INFO pre_bedrock_parity: Legacy enrichment progress: 150 articles
INFO pre_bedrock_parity: Legacy enrichment progress: 200 articles
INFO pre_bedrock_parity: Legacy enrichment pro

Legacy prepared records: 4127
Legacy source failures: 0


,records
platform,
opoint,4083
health_source,44


## 2. Run the new pipeline

The production entry point and Ebola YAML are used unchanged. Only runtime settings are overridden here: Bedrock extraction is forced off, Telegram/X remain off, and output is directed to this comparison run's folder.

In [6]:
new_config = load_project_config(MM_PIPELINE_ROOT / 'config' / 'projects' / 'ebola.yml')
new_config.extraction.enabled = False
new_config.output.backend = 'local'
new_config.output.base_uri = str(RUN_DIR / 'new_pipeline_output')
for source in new_config.sources:
    if source.platform == 'opoint':
        source.enabled = USE_OPOINT
    elif source.platform == 'health_source':
        source.enabled = USE_HEALTH
    elif source.platform in {'telegram', 'x'}:
        source.enabled = False

new_result = run_pipeline(
    new_config,
    window=WINDOW,
    logger=logger.getChild('new'),
)
new_df = pd.DataFrame([article.to_dict() for article in new_result.articles])
if new_df.empty:
    new_df = pd.DataFrame(columns=COMMON_COLUMNS + ['article_id', 'analysis_text'])
new_duplicate_count = int(new_df['article_id'].duplicated().sum())
new_path = RUN_DIR / 'new_master_data.jsonl'
new_df.to_json(new_path, orient='records', lines=True, force_ascii=False, date_format='iso')

print(f'New prepared records: {len(new_df)}')
print('New pipeline status:', 'succeeded' if new_result.succeeded else 'failed')
print('New source failures:', len(new_result.failures))
display(new_df['platform'].value_counts(dropna=False).rename('records').to_frame())

INFO pre_bedrock_parity.new.source.cdc_newsroom: Collecting with adapter rss
INFO pre_bedrock_parity.new.source.cdc_newsroom: Collected 0 record(s)
INFO pre_bedrock_parity.new.source.cdc_mmwr: Collecting with adapter rss
INFO pre_bedrock_parity.new.source.cdc_mmwr: Collected 2 record(s)
INFO pre_bedrock_parity.new.source.cdc_travel_notices: Collecting with adapter rss
INFO pre_bedrock_parity.new.source.cdc_travel_notices: Collected 3 record(s)
INFO pre_bedrock_parity.new.source.who_disease_outbreak_news: Collecting with adapter html_list
INFO pre_bedrock_parity.new.source.who_disease_outbreak_news: Collected 0 record(s)
INFO pre_bedrock_parity.new.source.who_africa: Collecting with adapter rss
INFO pre_bedrock_parity.new.source.who_africa: Collected 6 record(s)
INFO pre_bedrock_parity.new.source.africa_cdc: Collecting with adapter rss
INFO pre_bedrock_parity.new.source.africa_cdc: Collected 0 record(s)
INFO pre_bedrock_parity.new.source.ncdc_press: Collecting with adapter html_list
INF

New prepared records: 3981
New pipeline status: succeeded
New source failures: 0


,records
platform,
opoint,3977
health_source,4


## 3. Compare outputs

The comparison normalises whitespace, nulls, and timestamps first. It reports record-ID coverage separately from field-level differences so a source-list or filtering difference is not confused with a mapping difference.

In [7]:
COMPARE_FIELDS = [
    'platform', 'source_id', 'published_at', 'country', 'language', 'source_name',
    'author', 'headline', 'content', 'summary', 'url', 'search_id', 'analysis_text',
]

def normalise_for_comparison(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame.copy()
    for column in ['article_id', *COMPARE_FIELDS]:
        if column not in result.columns:
            result[column] = ''
    timestamps = pd.to_datetime(result['published_at'], errors='coerce', utc=True)
    result['published_at'] = timestamps.dt.strftime('%Y-%m-%dT%H:%M:%SZ').fillna('')
    for column in [c for c in ['article_id', *COMPARE_FIELDS] if c != 'published_at']:
        result[column] = result[column].fillna('').astype(str).str.strip()
    return result[['article_id', *COMPARE_FIELDS]].sort_values('article_id').reset_index(drop=True)

old_compare = normalise_for_comparison(legacy_df)
new_compare = normalise_for_comparison(new_df)
if not old_compare['article_id'].is_unique or not new_compare['article_id'].is_unique:
    raise AssertionError('Prepared datasets must have unique article IDs')

old_ids = set(old_compare['article_id'])
new_ids = set(new_compare['article_id'])
common_ids = sorted(old_ids & new_ids)
missing_from_new_ids = sorted(old_ids - new_ids)
only_in_new_ids = sorted(new_ids - old_ids)

old_by_id = old_compare.set_index('article_id')
new_by_id = new_compare.set_index('article_id')
difference_rows = []
for article_id in common_ids:
    for field in COMPARE_FIELDS:
        old_value = old_by_id.at[article_id, field]
        new_value = new_by_id.at[article_id, field]
        if old_value != new_value:
            difference_rows.append({
                'article_id': article_id,
                'field': field,
                'legacy_value': old_value,
                'new_value': new_value,
            })
field_differences = pd.DataFrame(
    difference_rows,
    columns=['article_id', 'field', 'legacy_value', 'new_value'],
)
missing_from_new = old_compare[old_compare['article_id'].isin(missing_from_new_ids)].copy()
only_in_new = new_compare[new_compare['article_id'].isin(only_in_new_ids)].copy()

source_counts = pd.concat([
    legacy_df.groupby(['platform', 'source_name'], dropna=False).size().rename('legacy'),
    new_df.groupby(['platform', 'source_name'], dropna=False).size().rename('new'),
], axis=1).fillna(0).astype(int).reset_index()
source_counts['difference'] = source_counts['new'] - source_counts['legacy']

missing_from_new.to_csv(RUN_DIR / 'missing_from_new.csv', index=False)
only_in_new.to_csv(RUN_DIR / 'only_in_new.csv', index=False)
field_differences.to_csv(RUN_DIR / 'field_differences.csv', index=False)
source_counts.to_csv(RUN_DIR / 'source_counts.csv', index=False)

print('Counts by platform and source:')
display(source_counts.sort_values(['platform', 'source_name']))
print('Field differences by field:')
display(field_differences['field'].value_counts().rename('differences').to_frame())
print('Sample records missing from the new output:')
display(missing_from_new[['article_id', 'source_name', 'headline']].head(20))
print('Sample records only in the new output:')
display(only_in_new[['article_id', 'source_name', 'headline']].head(20))

Counts by platform and source:


,platform,source_name,legacy,new,difference
0,health_source,CDC MMWR,12,0,-12
1,health_source,ECDC News,6,2,-4
2,health_source,Nigeria Centre for Disease Control,19,0,-19
3,health_source,WHO Regional Office for Africa,7,2,-5
4,opoint,20 Minutes,2,2,0
...,...,...,...,...,...
1437,opoint,thefact.ng,1,1,0
1438,opoint,therealmediang.com,4,2,-2
1439,opoint,İlke Haber Ajansı,1,1,0
1440,opoint,İlke Haber Ajansı - Ana safya,1,1,0


Field differences by field:


,differences
field,
summary,3928
content,2424
analysis_text,2424
published_at,27
source_name,27
headline,27
url,27
country,25
author,16


Sample records missing from the new output:


,article_id,source_name,headline
0,health_source_https://tools.cdc.gov/api/embed/...,CDC MMWR,First Clade Ib Monkeypox Virus Infection Repor...
1,health_source_https://tools.cdc.gov/api/embed/...,CDC MMWR,Case Finding Among and Comprehensive Managemen...
2,health_source_https://tools.cdc.gov/api/embed/...,CDC MMWR,Notes from the Field: Response to a Case of Tr...
3,health_source_https://tools.cdc.gov/api/embed/...,CDC MMWR,Notes from the Field: Outbreak of Ebola Diseas...
4,health_source_https://tools.cdc.gov/api/embed/...,CDC MMWR,Assessment of Risk to the U.S. Population from...
5,health_source_https://tools.cdc.gov/api/embed/...,CDC MMWR,Modeled Scenario Projections for the Ebola Dis...
6,health_source_https://tools.cdc.gov/api/embed/...,CDC MMWR,Legionnaires Disease Associated with a Private...
7,health_source_https://tools.cdc.gov/api/embed/...,CDC MMWR,Alcohol Consumption During Pregnancy Among Wom...
8,health_source_https://tools.cdc.gov/api/embed/...,CDC MMWR,QuickStats: Age-Adjusted Colorectal Cancer Dea...
9,health_source_https://tools.cdc.gov/api/embed/...,CDC MMWR,Notes from the Field: Characteristics and Moni...


Sample records only in the new output:


,article_id,source_name,headline
0,health_source_24312 at https://www.afro.who.int,WHO Regional Office for Africa,EBOLA BUNDIBUGYO VIRUS DISEASE OUTBREAK Democr...
1,health_source_24342 at https://www.afro.who.int,WHO Regional Office for Africa,Travel route of Kenya’s first Ebola case
2957,opoint_539353,Daily Nation - Kenya,How Uganda beat Ebola and what Kenya can learn


In [8]:
union_count = len(old_ids | new_ids)
field_difference_articles = (
    int(field_differences['article_id'].nunique()) if not field_differences.empty else 0
)
summary = {
    'window_start': START_TIME.isoformat(),
    'window_end': END_TIME.isoformat(),
    'legacy_records': len(legacy_df),
    'new_records': len(new_df),
    'common_article_ids': len(common_ids),
    'missing_from_new': len(missing_from_new_ids),
    'only_in_new': len(only_in_new_ids),
    'article_id_jaccard_percent': round(100 * len(common_ids) / max(union_count, 1), 2),
    'legacy_ids_found_in_new_percent': round(100 * len(common_ids) / max(len(old_ids), 1), 2),
    'common_articles_with_field_differences': field_difference_articles,
    'individual_field_differences': len(field_differences),
    'legacy_duplicate_ids_before_deduplication': legacy_duplicate_count,
    'new_duplicate_ids_after_pipeline': new_duplicate_count,
    'legacy_source_failures': legacy_failures,
    'new_source_failures': [failure.__dict__ for failure in new_result.failures],
    'new_fatal_errors': new_result.fatal_errors,
    'legacy_dataset': str(legacy_path),
    'new_dataset': str(new_path),
}
with (RUN_DIR / 'summary.json').open('w', encoding='utf-8') as handle:
    json.dump(summary, handle, indent=2, ensure_ascii=False, default=str)

display(pd.Series(summary, name='value').to_frame())
print('Full comparison written to:', RUN_DIR)

,value
window_start,2026-10-20T16:34:21+00:00
window_end,2026-10-27T16:34:21+00:00
legacy_records,4127
new_records,3981
common_article_ids,3978
missing_from_new,149
only_in_new,3
article_id_jaccard_percent,96.32
legacy_ids_found_in_new_percent,96.39
common_articles_with_field_differences,3932


Full comparison written to: C:\Users\TimSilman\Documents\other_projects\MM_Ebola_Tracker\mm_pipeline\parity_output\20261027T163421Z


## How to interpret the result

Start with `source_counts.csv`. A source that exists on only one side usually means the two source configurations differ. Then inspect `missing_from_new.csv` and `only_in_new.csv` for filtering, date-window, identifier, or fetch differences. Finally, use `field_differences.csv` to find mapping or content-enrichment differences among records that both pipelines collected.

Exact equality is not a reasonable pass condition for this live run because upstream pages can change between calls and the legacy OPoint function calculates its own current end time. Once the important differences have been understood, captured API/web responses can provide a deterministic regression test without altering the production architecture.